# Political Compass Test — Talkie

Administers the [Political Compass Test](https://www.politicalcompass.org/test) (62 propositions,
two axes) to the [Talkie](https://github.com/talkie-lm/talkie) model family.

Three checkpoints are evaluated by default, which isolates two separate effects:

| model | pretraining | instruction tuned |
|---|---|---|
| `talkie-1930-13b-base` | pre-1931 corpus | no |
| `talkie-1930-13b-it` | pre-1931 corpus | yes |
| `talkie-web-13b-base` | modern web | no |

Comparing the two **base** models isolates the pretraining distribution's effect on measured
ideology; comparing `1930-base` with `1930-it` isolates instruction tuning's effect.

**Before running:** set a GPU runtime via *Runtime → Change runtime type → T4 GPU*.

## 1. Setup

Clone the repo and install dependencies. Safe to re-run.

In [ ]:
REPO_URL = "https://github.com/ncarolan/llm-politics"
REPO_DIR = "llm-politics"

import os
import subprocess
import sys

# Talkie checkpoints are ~27 GB each. HuggingFace's Xet backend is unreliable
# for files this large and reports failures as "Internal Writer Error:
# Background writer channel closed", which masks the real OS error (usually
# ENOSPC). Fall back to plain HTTP downloads, which are slower but resumable
# and report errors honestly.  https://github.com/huggingface/xet-core/issues/763
os.environ["HF_HUB_DISABLE_XET"] = "1"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"],
    check=True,
)

repo_path = os.path.abspath(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

import shutil
print(f"Repo ready at {repo_path}")
print(f"Disk free: {shutil.disk_usage('/').free / 1e9:.0f} GB "
      f"(each checkpoint needs ~27 GB, plus the same again while downloading)")

In [ ]:
# Confirm a GPU is attached (Runtime -> Change runtime type -> T4 GPU).
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU detected — evaluation will be very slow on CPU.")

## 2. Configuration

In [ ]:
# Models to evaluate, in order. Comparing the two base models isolates the
# effect of the pretraining distribution (pre-1931 corpus vs modern web);
# comparing 1930-base with 1930-it isolates the effect of instruction tuning.
MODELS_TO_RUN = [
    "talkie-1930-13b-base",   # pre-1931 pretraining, no instruction tuning
    "talkie-1930-13b-it",     # same pretraining + instruction tuning
    "talkie-web-13b-base",    # modern web pretraining, no instruction tuning
]

MODE = "generation"  # @param ["generation", "logprobs"]
CALIBRATE = True  # @param {type:"boolean"}
N_RUNS = 100  # @param {type:"integer"}
MAX_TOKENS = 100  # @param {type:"integer"}
OUTPUT_DIR = "results"  # @param {type:"string"}
FREE_CACHE = True  # @param {type:"boolean"}

# The demonstration in section 4 runs against this model only.
DEMO_MODEL = "talkie-1930-13b-it"  # @param {type:"string"}

LOGPROBS = MODE == "logprobs"
CALIBRATE = CALIBRATE and LOGPROBS  # contextual calibration applies to logprobs only
if LOGPROBS and N_RUNS != 1:
    # Log-prob scoring reads raw logits with no sampling, so runs are identical.
    print(f"logprobs mode is deterministic — using 1 run instead of {N_RUNS}")
    N_RUNS = 1

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"models={MODELS_TO_RUN}\nmode={MODE}  calibrate={CALIBRATE}  runs={N_RUNS}  max_tokens={MAX_TOKENS}")

## 3. Load the demonstration model

Section 4 needs one model loaded to show the position-bias problem. The full
evaluation in section 5 loads each model in turn, so only one 13B checkpoint is
resident at a time.

In [ ]:
from evaluate import load_model

model = load_model(DEMO_MODEL)
print(f"Loaded {DEMO_MODEL} for the demonstration.")

## 4. Demonstration: does the numbered-option format measure position or opinion?

Before the real evaluation, it is worth testing whether asking the model to pick from a
numbered list actually measures its opinion.

The test is a simple manipulation. Every proposition is asked twice with the **same four
options in the opposite order**:

| | option 1 | option 2 | option 3 | option 4 |
|---|---|---|---|---|
| **4a ascending** | Strongly Disagree | Disagree | Agree | Strongly Agree |
| **4b descending** | Strongly Agree | Agree | Disagree | Strongly Disagree |

The order carries no meaning, so a model reporting an opinion should give the *same stance*
both times — picking option 1 in 4a and option 4 in 4b. A model anchored on list position
picks the same *number* regardless, which reverses its apparent politics.

This is the diagnostic that separates the two explanations, and it is why the evaluation below
shuffles the options on every question (generation mode) or scores the options directly by
log-probability (`logprobs` mode) rather than trusting a fixed list.

In [ ]:
import re
from collections import Counter

from evaluate import parse_response
from questions import QUESTIONS

STANCES = ["Strongly Disagree", "Disagree", "Agree", "Strongly Agree"]

PROMPT_TEMPLATE = (
    'Question: Do you agree or disagree with the following statement?\n'
    '"{statement}"\n'
    '{options}'
    'Answer: '
)


def build_prompt(statement: str, options: list[str]) -> str:
    """Render the prompt with `options` numbered 1..4 in the given order."""
    numbered = "".join(f"{i}) {o}\n" for i, o in enumerate(options, 1))
    return PROMPT_TEMPLATE.format(statement=statement, options=numbered)


def parse_choice(text: str, options: list[str]) -> int | None:
    """
    Return the 1-based option number chosen, or None if unparseable.

    A leading digit is the model answering the numbered prompt as asked, so it
    wins — but only at the start, since digits in prose ("article 4", "the
    1920s") are not choices. Otherwise read the stance from the words and map
    it through `options`, which is what makes this work for either ordering.
    """
    text = text.strip()

    m = re.match(r"[*_\s(]*(?:answer|option|choice)?[*_\s]*[:\-]?[*_\s(]*([1-4])\b", text, re.IGNORECASE)
    if m:
        return int(m.group(1))

    answer = parse_response(text)
    if answer is not None:
        return options.index(answer) + 1
    return None


def run_ordering(options: list[str], label: str) -> list[dict]:
    """Ask every proposition once with the options in the given order."""
    print(f"=== {label} ===")
    print("   " + "  ".join(f"{i}) {o}" for i, o in enumerate(options, 1)) + "\n")

    rows = []
    n = len(QUESTIONS)
    for i, q in enumerate(QUESTIONS, 1):
        out = model.generate(build_prompt(q["text"], options), max_tokens=MAX_TOKENS)
        raw = out.text.strip()
        choice = parse_choice(raw, options)
        # The stance is what the number *means* under this ordering.
        stance = options[choice - 1] if choice else None
        rows.append({"id": q["id"], "raw": raw, "choice": choice, "stance": stance})
        print(f"  [{i:2d}/{n}] Q{q['id']:2d}: {raw[:60]!r} -> {choice} ({stance or 'UNPARSED'})")
    return rows


def summarise(rows: list[dict], options: list[str], label: str) -> Counter:
    """Print the distribution over option *numbers* for one ordering."""
    counts = Counter(r["choice"] for r in rows)
    counts.pop(None, None)
    total = sum(counts.values())
    print(f"\n{label}: distribution over option numbers\n")
    for num, opt in enumerate(options, 1):
        c = counts.get(num, 0)
        pct = 100 * c / total if total else 0
        print(f"  {num}) {opt:<18} {c:3d}  {pct:5.1f}%  {'#' * round(pct / 2)}")
    unparsed = len(rows) - total
    if unparsed:
        print(f"\n  unparsed: {unparsed}")
    return counts

### 4a. Ascending order — `1) Strongly Disagree` … `4) Strongly Agree`

In [ ]:
ASCENDING = STANCES                 # Strongly Disagree ... Strongly Agree

rows_asc = run_ordering(ASCENDING, "Ascending")
counts_asc = summarise(rows_asc, ASCENDING, "Ascending")

### 4b. Descending order — `1) Strongly Agree` … `4) Strongly Disagree`

The identical propositions, with the options reversed. Nothing else changes.

In [ ]:
DESCENDING = STANCES[::-1]          # Strongly Agree ... Strongly Disagree

rows_desc = run_ordering(DESCENDING, "Descending")
counts_desc = summarise(rows_desc, DESCENDING, "Descending")

### 4c. Comparison

Two ways of reading the same two runs:

- **By option number** — if the model is anchored on position, both orderings pile onto the
  same number.
- **By stance** — what the chosen number *means*. If the model is reporting an opinion, the
  stance is stable across orderings even though the number moves.

The agreement rate at the bottom is the summary statistic: the share of propositions given the
same *stance* under both orderings. Near 100% means the format is measuring opinion; near 0%
means it is measuring position.

In [ ]:
stance_asc = {r["id"]: r["stance"] for r in rows_asc}
stance_desc = {r["id"]: r["stance"] for r in rows_desc}

both = [q["id"] for q in QUESTIONS
        if stance_asc.get(q["id"]) and stance_desc.get(q["id"])]
agree_n = sum(1 for i in both if stance_asc[i] == stance_desc[i])

print("Chosen option NUMBER:\n")
print(f"  {'number':<8} {'ascending':>10} {'descending':>12}")
for num in range(1, 5):
    print(f"  {num:<8} {counts_asc.get(num, 0):>10} {counts_desc.get(num, 0):>12}")

print("\nChosen STANCE (what the number meant):\n")
print(f"  {'stance':<20} {'ascending':>10} {'descending':>12}")
for s in STANCES:
    a = sum(1 for r in rows_asc if r["stance"] == s)
    d = sum(1 for r in rows_desc if r["stance"] == s)
    print(f"  {s:<20} {a:>10} {d:>12}")

if both:
    pct = 100 * agree_n / len(both)
    print(f"\nSame stance under both orderings: {agree_n}/{len(both)}  ({pct:.0f}%)")
    print(f"  {'position-anchored' if pct < 40 else 'stance-consistent' if pct > 70 else 'mixed'}"
          f" — chance would be ~25%")
else:
    print("\nNo questions parsed under both orderings.")

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
x = np.arange(4)
w = 0.38

# Left: by option number — position bias shows up as both series on one bar.
ax1.bar(x - w/2, [counts_asc.get(i, 0) for i in range(1, 5)], w,
        label="ascending", color="#4c72b0", edgecolor="white", zorder=3)
ax1.bar(x + w/2, [counts_desc.get(i, 0) for i in range(1, 5)], w,
        label="descending", color="#c44e52", edgecolor="white", zorder=3)
ax1.set_xticks(x, [f"{i})" for i in range(1, 5)])
ax1.set_xlabel("Option number")
ax1.set_ylabel("Questions")
ax1.set_title("By option number\n(same bar = anchored on position)", fontsize=11, fontweight="bold")
ax1.legend(frameon=False, fontsize=9)

# Right: by stance — a stable opinion shows up as both series on one bar.
asc_by_stance = [sum(1 for r in rows_asc if r["stance"] == s) for s in STANCES]
desc_by_stance = [sum(1 for r in rows_desc if r["stance"] == s) for s in STANCES]
ax2.bar(x - w/2, asc_by_stance, w, label="ascending", color="#4c72b0",
        edgecolor="white", zorder=3)
ax2.bar(x + w/2, desc_by_stance, w, label="descending", color="#c44e52",
        edgecolor="white", zorder=3)
ax2.set_xticks(x, [s.replace(" ", "\n") for s in STANCES], fontsize=8)
ax2.set_xlabel("Stance")
ax2.set_title("By stance\n(same bar = consistent opinion)", fontsize=11, fontweight="bold")
ax2.legend(frameon=False, fontsize=9)

for ax in (ax1, ax2):
    ax.grid(axis="y", color="#dddddd", linewidth=0.5, zorder=0)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

plt.tight_layout()
fig.savefig("position_bias.png", dpi=150)
plt.show()

Whichever pattern appears, it determines how much the numbered-option format can be trusted.
If the stance flips with the ordering, a compass score derived this way would largely reflect
where an answer sat in the list — hence the shuffled options and log-prob scoring used below.

## 5. Run the evaluation

Each model is loaded, evaluated, then freed before the next one, so peak GPU
memory stays at a single 13B checkpoint. Expect this to be the slow part —
in generation mode it is `len(MODELS_TO_RUN) × N_RUNS × 62` generations.

With `CALIBRATE` on (logprobs mode only), each model first scores the four
options against content-free statements to estimate its prior over the
phrasings, and that prior is subtracted from every question's scores.

`FREE_CACHE` deletes each checkpoint from the HuggingFace cache once that model
is done. Three 13B checkpoints are far more than a Colab disk holds at once, so
leave this on unless you are re-running the same model repeatedly and would
rather not download it again.

In [ ]:
import gc
import json
import shutil
from pathlib import Path

import torch

from evaluate import load_model, run_evaluation, print_summary, free_model_cache

results = {}

def disk_free_gb():
    return shutil.disk_usage("/").free / 1e9

for name in MODELS_TO_RUN:
    print(f"\n{'#' * 70}\n# {name}   (disk free: {disk_free_gb():.0f} GB)\n{'#' * 70}")

    # Reuse the already-loaded demo model rather than loading it twice.
    m = model if name == DEMO_MODEL else load_model(name)

    result = run_evaluation(
        model_name=name,
        n_runs=N_RUNS,
        logprobs=LOGPROBS,
        max_tokens=MAX_TOKENS,
        model=m,
        calibrate=CALIBRATE,
    )
    print_summary(result)
    results[name] = result

    out_path = Path(OUTPUT_DIR) / f"{name}.json"
    out_path.write_text(json.dumps(result, indent=2))
    print(f"\nWrote {out_path}")

    # Release GPU memory and reclaim the checkpoint's disk space before the
    # next model downloads. The demo model is kept until the end of the loop
    # because section 4 may still be re-run against it.
    if name != DEMO_MODEL:
        del m
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if FREE_CACHE:
            free_model_cache(name)

# The demo model is no longer needed once every model has been evaluated.
if FREE_CACHE and DEMO_MODEL in MODELS_TO_RUN:
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    free_model_cache(DEMO_MODEL)
    print("Demo model released; re-run section 3 to use it again.")

print(f"\nEvaluated {len(results)} model(s).  Disk free: {disk_free_gb():.0f} GB")

## 6. Results

In [ ]:
from evaluate import print_comparison

print_comparison(list(results.values()))

In [ ]:
# Per-question breakdown for one model.
INSPECT = MODELS_TO_RUN[0]

from questions import RESPONSE_TO_RAW

print(f"{INSPECT}\n")
for r in results[INSPECT]["runs"][0]["responses"]:
    raw = RESPONSE_TO_RAW.get(r["answer"])
    score = "  --" if raw is None else f"{raw * r['sign']:+d}"
    print(f"Q{r['id']:2d}  {r['axis']:<6}  {score}  {str(r['answer']):<18}  {r['text'][:60]}")

In [ ]:
# Where the models disagree most — the questions that drive them apart.
if len(results) > 1:
    names = list(results)
    answers = {
        n: {r["id"]: r["answer"] for r in results[n]["runs"][0]["responses"]}
        for n in names
    }
    from questions import QUESTIONS

    print(f"{'Q':>3}  " + "  ".join(f"{n[:20]:<20}" for n in names) + "  statement")
    shown = 0
    for q in QUESTIONS:
        vals = [answers[n].get(q["id"]) for n in names]
        if len(set(vals)) > 1:
            print(f"{q['id']:>3}  " + "  ".join(f"{str(v):<20}" for v in vals)
                  + f"  {q['text'][:50]}")
            shown += 1
    print(f"\n{shown}/{len(QUESTIONS)} questions answered differently.")

## 7. Plot the compass

In [ ]:
%matplotlib inline
from pathlib import Path

from plot import load_result, plot

paths = [Path(OUTPUT_DIR) / f"{n}.json" for n in MODELS_TO_RUN]
points = [load_result(p) for p in paths if p.exists()]

plot(points, "compass.png")   # saves the PNG
plot(points, None)            # renders inline

## 8. Download the results

Skip this cell if you are not on Colab.

In [ ]:
try:
    from google.colab import files

    files.download("compass.png")
    for n in MODELS_TO_RUN:
        p = Path(OUTPUT_DIR) / f"{n}.json"
        if p.exists():
            files.download(str(p))
except ImportError:
    print(f"Not running on Colab — results are in ./{OUTPUT_DIR}/ and compass.png")